# Predict with PVT-MoE
Loads `model.pt` and reports metrics on the validation set.

In [ ]:
import pandas as pd, torch
from torch.utils.data import DataLoader
from dataset import PVTDataset
from model import PVTMoE
from engine import predict, metrics

ROOT, DEVICE = "data", "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("model.pt", map_location=DEVICE)
model = PVTMoE(pretrained=False).to(DEVICE)
model.load_state_dict(ckpt["model"])

In [ ]:
df = pd.read_csv(f"{ROOT}/table.csv")
va = df[df.split == "val"]
dl = DataLoader(PVTDataset(va, ROOT, ckpt["mean"], ckpt["std"]), 8)
p, y = predict(model, dl, DEVICE)
pd.DataFrame(dict(id=va["id"].values, label=y, prob_neoplastic=p)).to_csv("predictions.csv", index=False)
print(metrics(p, y))